# Introduction
This notebook looks at testing api calls and authentication in accessing my App within the isthereanydeal website. <br>
App: https://isthereanydeal.com/apps/3847/ <br>
Documentation: https://docs.isthereanydeal.com/

# API Test

First let's test how we can query the api using our apikey/secrets

In [1]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd

In [2]:
load_dotenv()

itad_api_key = os.getenv('itad_api_key')

# OAuth Endpoints
base_url = "https://api.isthereanydeal.com"

url = f"{base_url}/games/lookup/v1"

params = {
    'title': 'Megabonk',
    'key': itad_api_key
}

response = requests.get(url, params=params)
response.raise_for_status()
game_data = response.json()
game_data_df = pd.DataFrame(game_data.get('game', []))
game_data_df

,id,slug,title,type,mature,assets
boxart,01940f12-0f62-7025-babd-068ff1881519,megabonk,Megabonk,game,False,https://assets.isthereanydeal.com/01940f12-0f6...
banner145,01940f12-0f62-7025-babd-068ff1881519,megabonk,Megabonk,game,False,https://assets.isthereanydeal.com/01940f12-0f6...
banner300,01940f12-0f62-7025-babd-068ff1881519,megabonk,Megabonk,game,False,https://assets.isthereanydeal.com/01940f12-0f6...
banner400,01940f12-0f62-7025-babd-068ff1881519,megabonk,Megabonk,game,False,https://assets.isthereanydeal.com/01940f12-0f6...
banner600,01940f12-0f62-7025-babd-068ff1881519,megabonk,Megabonk,game,False,https://assets.isthereanydeal.com/01940f12-0f6...


In [7]:
url = f"{base_url}/games/history/v2"

id = game_data_df['id'].unique()
    
params = {
    'id': id,
    'country': 'AU',
    'key': itad_api_key
}

response = requests.get(url, params=params)
response.raise_for_status()
historical_data = response.json()
historical_data_df = pd.DataFrame(historical_data)
print(historical_data)

[{'timestamp': '2025-10-02T19:18:27+02:00', 'shop': {'id': 61, 'name': 'Steam'}, 'deal': {'price': {'amount': 14.5, 'amountInt': 1450, 'currency': 'AUD'}, 'regular': {'amount': 14.5, 'amountInt': 1450, 'currency': 'AUD'}, 'cut': 0}}, {'timestamp': '2025-09-18T20:18:26+02:00', 'shop': {'id': 61, 'name': 'Steam'}, 'deal': {'price': {'amount': 12.32, 'amountInt': 1232, 'currency': 'AUD'}, 'regular': {'amount': 14.5, 'amountInt': 1450, 'currency': 'AUD'}, 'cut': 15}}]


In [9]:
# testing json to df
game_data_df_test = pd.json_normalize(game_data)
game_data_df_test

,found,game.id,game.slug,game.title,game.type,game.mature,game.assets.boxart,game.assets.banner145,game.assets.banner300,game.assets.banner400,game.assets.banner600
0,True,01940f12-0f62-7025-babd-068ff1881519,megabonk,Megabonk,game,False,https://assets.isthereanydeal.com/01940f12-0f6...,https://assets.isthereanydeal.com/01940f12-0f6...,https://assets.isthereanydeal.com/01940f12-0f6...,https://assets.isthereanydeal.com/01940f12-0f6...,https://assets.isthereanydeal.com/01940f12-0f6...


In [10]:
historical_data_df_test = pd.json_normalize(historical_data)
historical_data_df_test

,timestamp,shop.id,shop.name,deal.price.amount,deal.price.amountInt,deal.price.currency,deal.regular.amount,deal.regular.amountInt,deal.regular.currency,deal.cut
0,2025-10-02T19:18:27+02:00,61,Steam,14.50,1450,AUD,14.5,1450,AUD,0
1,2025-09-18T20:18:26+02:00,61,Steam,12.32,1232,AUD,14.5,1450,AUD,15


# Testing api_calls functions

In [7]:
from api_calls import itadapi

balatro = itadapi.lookup_game(title = 'Balatro')
balatro

,found,game_id,game_slug,game_title,game_type,game_mature,game_assets_boxart,game_assets_banner145,game_assets_banner300,game_assets_banner400,game_assets_banner600
0,True,018d937f-700e-7161-9c8d-5423af1b7c99,balatro,Balatro,game,False,https://assets.isthereanydeal.com/018d937f-700...,https://assets.isthereanydeal.com/018d937f-700...,https://assets.isthereanydeal.com/018d937f-700...,https://assets.isthereanydeal.com/018d937f-700...,https://assets.isthereanydeal.com/018d937f-700...


In [4]:
balatro.found

0    False
Name: found, dtype: bool

In [4]:
from api_calls import itadapi

balatro = itadapi.get_historical_prices("Balatro", 'AU')
balatro

,timestamp,shop_id,shop_name,deal_price_amount,deal_price_amountInt,deal_price_currency,deal_regular_amount,deal_regular_amountInt,deal_regular_currency,deal_cut
0,2025-10-24T01:21:03+02:00,6,Fanatical,21.95,2195,AUD,21.95,2195,AUD,0
1,2025-10-23T19:26:12+02:00,37,Humble Store,19.75,1975,AUD,21.95,2195,AUD,10
2,2025-10-13T10:25:15+02:00,6,Fanatical,19.75,1975,AUD,21.95,2195,AUD,10
3,2025-10-09T03:36:00+02:00,2,AllYouPlay,14.99,1499,USD,14.99,1499,USD,0
4,2025-10-09T03:16:49+02:00,64,WinGameStore,14.99,1499,USD,14.99,1499,USD,0
...,...,...,...,...,...,...,...,...,...,...
81,2025-07-25T06:03:40+02:00,64,WinGameStore,14.99,1499,USD,14.99,1499,USD,0
82,2025-07-25T06:03:40+02:00,47,MacGameStore,14.99,1499,USD,14.99,1499,USD,0
83,2025-07-25T06:03:40+02:00,61,Steam,19.75,1975,AUD,21.95,2195,AUD,10
84,2025-07-25T06:03:40+02:00,37,Humble Store,21.95,2195,AUD,21.95,2195,AUD,0
